# Cassegrain Slab Numerical Simulations
This notebook prepares and launches Tidy3D transport simulations for Local Self Uniform Structures used in the experimental setup by Abraham A. Gonzalez.

The cells below are organized to:
1. load the scientific and project-specific tooling,
2. define the angular sampling and sweep parameters,
3. configure a partial illuminating-cone experiment,
4. iterate over structure files and candidate cuts before running or preparing simulations.

In [1]:
# Notebook setup and core scientific imports
%load_ext autoreload
%autoreload 2
%matplotlib inline
import scipy.io as sio
from dataclasses import dataclass
from typing import List, Tuple
import os
from dotenv import load_dotenv

# Load environment variables such as API credentials.
load_dotenv()

# Tidy3D and numerical utilities used throughout the notebook.
import tidy3d as td
from tidy3d import web
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

## Environment and project imports
These first cells initialize the notebook environment and bring the local automation helpers into scope so the simulation setup can be expressed in a compact way.

In [2]:
import sys
import os

# Add the local project root so the custom automation package can be imported.
sys.path.append(os.path.abspath(fr'H:\phd stuff\tidy3d'))

# Import the structure-loading and simulation helpers used below.
from AutomationModule import *

import AutomationModule as AM

In [3]:
# Read the Tidy3D API key from the environment for authenticated runs.
tidy3dAPI = os.environ["API_TIDY3D_KEY"]


In [4]:
# Generate a dense spherical sampling grid for far-field angular projections.
sphere, phi, theta = AM.get_sphere(1000) #Refer to the tools section

## Angular sampling and sweep parameters
This section defines the wavelength range, the angular projection grid, and the normalized geometric cuts that will be tested for each structure file.

In [ ]:
# Sweep over the wavelength interval of interest for the slab response.
lambdas = np.array([8, 2.5])

# Project the far field onto a very distant observation sphere.
r_proj = 1e6  # radial distance away from the origin at which to project fields
theta_proj = theta
phi_proj = phi

# Convert target physical sizes into normalized cut conditions for the structure loader.
sizes_exp = np.array([12])
unit_cell_size = 11.44
cuts = sizes_exp / unit_cell_size

In [6]:
# Directory containing the candidate structure files to simulate.
folder_path = rf"./Structures"

# Label used to organize generated data products on disk.
project_name = "20260310 Far Field Transmission LSU ff_21p72_5x_size"

# Global simulation controls shared by every run in the sweep.
runtime_ps = 25e-12
min_steps_per_lambda = 20
ref = False

### Partial illuminating-cone simulation
This numerical experimient is configured to simulate only one section of the illuminating cone in the Cassegrain experiment setup using a focused Gaussian Beaam.

The combination of `direction`, `cut_condition`, `far_field_settings`, and `gaussian_params` selects a single illumination geometry: one incidence direction, one structure cut, and one angular projection window for the emitted or collected field.


## Simulation execution
The final run cell loops over the available structures and normalized cuts, builds one partial-cone illumination case at a time, and prepares the corresponding far-field simulation outputs.

In [ ]:
# Warning: this sweep can cost more than 100 credits if fully executed.
ref = False
h5_bg = None

# Loop over one incidence direction and over all available slab files and cut values.
for direction in ["z"]:
    for f, filename in enumerate(os.listdir(folder_path)):
        for cut in cuts:
            # Only process saved structure files.
            if not (Path(filename).suffix == ".h5"):
                continue

            if os.path.isfile(os.path.join(folder_path, filename)):
                file = os.path.join(folder_path, filename)

                # Build a Gaussian-illuminated partial-cone Cassegrain configuration for this structure.
                structure_1 = AM.loadAndRunStructure( #Our module is fully automated and handles geometry loading, simulation setup, and execution
                    key=tidy3dAPI,
                    file_path=file,
                    direction=direction,
                    lambda_range=lambdas,
                    box_size=unit_cell_size,
                    runtime_ps=runtime_ps,
                    min_steps_per_lambda=min_steps_per_lambda,
                    scaling=1,
                    shuoff_condtion=1e-20,
                    verbose=True,
                    multiplicate_size=True,
                    multiplication_factor= 5,# We create a large slab by putting together 5 copies of the original structure in a grid. 
                                             # This is necessary to ensure that the Gaussian source is fully contained within the structure and to mitigate finite-size effects on the far-field projections.
                    monitors=["flux", "far_field"], # We monitor both near-field flux and far-field projections in this sweep, but the latter is the main quantity of interest.
                    flux_monitor_position=12.5,
                    cell_size_manual=35,
                    freqs=250,
                    far_field_settings={"r": r_proj, "theta": theta_proj, "phi": phi_proj, "window_size": 0.25},
                    cut_condition=cut,
                    source="gaussian",
                    gaussian_params={"waist_radius": 14, "waist_distance": -16, "theta": 22.5 * np.pi / 180, "phi": 0, "position_x": -19, "size": 28},
                    absorbers=130,
                    boundaries="absorbers",
                    use_permittivity=False,
                    sim_name=rf"{Path(filename).stem}_size_{cut}" + (rf"_bg_{h5_bg:.2f}" if h5_bg else ""),
                    h5_bg=h5_bg,  # Optional homogeneous background in the simulation.
                    # ref_only=True,
                )

                # Inspect the generated simulation geometry before launching a remote run.
                structure_1.sim.plot_3d()

                # Check whether a result description already exists on disk for this setup.
                file_desc = rf"H:\phd stuff\tidy3d\data\{project_name}_perm_{structure_1.permittivity_value:.2f}\z_incidence\{structure_1.sim_name}.txt"
                if os.path.exists(file_desc):
                    print("Exist!")
                else:
                    print("Creating...")
                    # Warning: this sweep can cost more than 100 credits if fully executed.
                    # structure_1.run_sim(run_free=False, load=False, add_ref=ref, folder_description=rf"{project_name}_perm_{structure_1.permittivity_value:.2f}", monitor=True)
                    if ref:
                        ref = False



Configured successfully.


16:09:42 W. Europe Standard Time WARNING: Structure at 'structures[0]' has      
                                 bounds that extend exactly to simulation edges.
                                 This can cause unexpected behavior. If         
                                 intending to extend the structure to infinity  
                                 along one dimension, use td.inf as a size      
                                 variable instead to make this explicit.        

                                 WARNING: Suppressed 19 WARNING messages.       

                                 WARNING: Field projection monitor 'far_field'  
                                 has observation points set up such that the    
                                 monitor is projecting backwards with respect to
                                 its 'normal_dir'. If this was not intentional, 
                                 please take a look at the documentation        
                                 associated with this type of projection monitor
                                 to check how the observation point coordinate  
                                 system is defined.                             